# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joudData/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### Unit of Analysis

One row represents one content item, identified by `content_hash_id`, in the `dim_content` table.

The analysis uses content-level characteristics to identify content archetypes.

### Time Window

March 2026 (`month=2026-03`) is used as the development window for the daily performance data.

June 2026 is reserved as a sealed test month and is not used for development or label logic.

*One row = one what, over which dates? State it, then verify it below.*

## 2. Fields: feature / label / context / excluded


Features:
- content_type
- word_count
- char_count
- keyword_token_count
- category_count

Label:
- No predefined label. The goal is to discover content archetypes through clustering.

Context:
- content_hash_id
- client_hash_id
- content_created_date
- content_updated_date

Excluded:
- keyword_hash_id and url_hash_id are excluded because they are identifiers, not meaningful content characteristics. Using them could introduce noise into clustering.


*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [14]:

grain_check = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content_items,
    COUNT(*) FILTER (
        WHERE content_hash_id IS NULL
    ) AS missing_content_ids,
    COUNT(*) FILTER (
        WHERE content_type IS NULL
    ) AS missing_content_type,
    COUNT(*) FILTER (
        WHERE word_count IS NULL
    ) AS missing_word_count,
    COUNT(*) FILTER (
        WHERE char_count IS NULL
    ) AS missing_char_count,
    COUNT(*) FILTER (
        WHERE keyword_token_count IS NULL
    ) AS missing_keyword_token_count,
    COUNT(*) FILTER (
        WHERE category_count IS NULL
    ) AS missing_category_count
FROM content
""").df()

display(grain_check)



,total_rows,unique_content_items,missing_content_ids,missing_content_type,missing_word_count,missing_char_count,missing_keyword_token_count,missing_category_count
0,519606,519606,0,0,177768,177768,0,0



The grain check verifies that each row represents one content item. It also checks missing values across the five selected features.


In [12]:

date_check = con.execute("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM performance
""").df()

display(date_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31




The March 2026 performance slice contains 9,841,378 rows, covering March 1 through March 31, 2026.



In [15]:


availability_check = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
        OR ga4_data_available IS TRUE
    ) AS rows_with_available_data
FROM performance
""").df()

display(availability_check)



,total_rows,gsc_available_rows,ga4_available_rows,rows_with_available_data
0,9841378,3611061,413966,3660680



The availability check uses IS TRUE to identify rows with available GSC and GA4 data. A total of 3,660,680 rows have at least one available data source.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*




- **Unbalanced history:** Content items have different amounts of historical performance data, which makes direct comparisons less reliable.

- **GSC-only early rows:** Some early records have GSC data available but lack GA4 data, so the performance information is incomplete.

- **Window overlaps:** Overlapping time windows may cause data leakage if information from the outcome period is included in the features.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Leakage Trap – Feature**

For the leakage experiment, a proxy label is created from GA4 sessions to distinguish high-session and low-session content. A label-derived feature will then be added intentionally to demonstrate data leakage.


In [23]:

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

# 1. Keep rows where GA4 data is available
ga4_data = performance_df[
    performance_df["ga4_data_available"].eq(True)
]

# 2. Aggregate sessions for each content item
content_sessions = (
    ga4_data.groupby("content_hash_id")["ga4_sessions"]
    .sum()
    .reset_index()
)

# 3. Join content features with the proxy label source
leakage_df = content_df[
    ["content_hash_id", "word_count", "char_count",
     "keyword_token_count", "category_count"]
].merge(content_sessions, on="content_hash_id", how="inner")

# 4. Create a proxy label
threshold = leakage_df["ga4_sessions"].median()
y = (leakage_df["ga4_sessions"] >= threshold).astype(int)

# 5. Select content features (exclude GA4 sessions)
X = leakage_df[
    ["word_count", "char_count",
     "keyword_token_count", "category_count"]
]

# 6. Split once, using the same rows for both experiments
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 7. Honest model: no leaked feature
honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(max_depth=5, random_state=42)
)
honest_model.fit(X_train, y_train)
honest_score = accuracy_score(y_test, honest_model.predict(X_test))

# 8. Intentionally add one label-derived feature
X_train_leak = X_train.copy()
X_test_leak = X_test.copy()

X_train_leak["label_leak"] = y_train
X_test_leak["label_leak"] = y_test

leak_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(random_state=42)
)
leak_model.fit(X_train_leak, y_train)
leaked_score = accuracy_score(
    y_test, leak_model.predict(X_test_leak)
)

print("Honest score:", round(honest_score, 4))
print("Leaked score:", round(leaked_score, 4))

# 9. Remove the leaked feature and retain the honest result
X_train_leak = X_train_leak.drop(columns=["label_leak"])
X_test_leak = X_test_leak.drop(columns=["label_leak"])

final_honest_score = honest_score
print("Final honest score:", round(final_honest_score, 4))





Honest score: 0.667
Leaked score: 1.0
Final honest score: 0.667



### Leakage Trap Results

A proxy label was created using aggregated GA4 sessions to distinguish high-session and low-session content.

A label-derived feature was intentionally added to demonstrate data leakage.

The honest model was evaluated without the leaked feature. The leaked model was evaluated after adding the label-derived column.

The leaked feature was then removed, and the honest score was retained.




In [24]:

# Remove the leaked feature
X_leaked = X_leaked[:, :-1]

# Keep the honest score
final_honest_score = honest_score

print("Final honest accuracy:", round(final_honest_score, 4))


Final honest accuracy: 0.667



### Leakage Lesson

The leaked feature was removed, and the honest accuracy of 0.6669 was retained.

Features derived from the target label must not be used during model training because they lead to misleading evaluation results.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.